In [26]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_excel("AirQualityUCI.xlsx")


In [28]:
df

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2004-03-10,18:00:00,2.6,1360.00,150,11.881723,1045.50,166.0,1056.25,113.0,1692.00,1267.50,13.600,48.875001,0.757754
1,2004-03-10,19:00:00,2.0,1292.25,112,9.397165,954.75,103.0,1173.75,92.0,1558.75,972.25,13.300,47.700000,0.725487
2,2004-03-10,20:00:00,2.2,1402.00,88,8.997817,939.25,131.0,1140.00,114.0,1554.50,1074.00,11.900,53.975000,0.750239
3,2004-03-10,21:00:00,2.2,1375.50,80,9.228796,948.25,172.0,1092.00,122.0,1583.75,1203.25,11.000,60.000000,0.786713
4,2004-03-10,22:00:00,1.6,1272.25,51,6.518224,835.50,131.0,1205.00,116.0,1490.00,1110.00,11.150,59.575001,0.788794
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9352,2005-04-04,10:00:00,3.1,1314.25,-200,13.529605,1101.25,471.7,538.50,189.8,1374.25,1728.50,21.850,29.250000,0.756824
9353,2005-04-04,11:00:00,2.4,1162.50,-200,11.355157,1027.00,353.3,603.75,179.2,1263.50,1269.00,24.325,23.725000,0.711864
9354,2005-04-04,12:00:00,2.4,1142.00,-200,12.374538,1062.50,293.0,603.25,174.7,1240.75,1092.00,26.900,18.350000,0.640649
9355,2005-04-04,13:00:00,2.1,1002.50,-200,9.547187,960.50,234.5,701.50,155.7,1041.00,769.75,28.325,13.550000,0.513866


In [29]:
df.replace(-200, np.nan, inplace=True)

In [30]:
print(df.isnull().sum())

Date                0
Time                0
CO(GT)           1683
PT08.S1(CO)       366
NMHC(GT)         8443
C6H6(GT)          366
PT08.S2(NMHC)     366
NOx(GT)          1639
PT08.S3(NOx)      366
NO2(GT)          1642
PT08.S4(NO2)      366
PT08.S5(O3)       366
T                 366
RH                366
AH                366
dtype: int64


In [31]:
df.fillna(df.mean(numeric_only=True), inplace=True)

In [32]:
# Combine Date and Time
df['Datetime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'].astype(str))

# Drop old columns
df.drop(['Date', 'Time'], axis=1, inplace=True)

print(df.head())

   CO(GT)  PT08.S1(CO)  NMHC(GT)   C6H6(GT)  PT08.S2(NMHC)  NOx(GT)  \
0     2.6      1360.00     150.0  11.881723        1045.50    166.0   
1     2.0      1292.25     112.0   9.397165         954.75    103.0   
2     2.2      1402.00      88.0   8.997817         939.25    131.0   
3     2.2      1375.50      80.0   9.228796         948.25    172.0   
4     1.6      1272.25      51.0   6.518224         835.50    131.0   

   PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)      T         RH  \
0       1056.25    113.0       1692.00      1267.50  13.60  48.875001   
1       1173.75     92.0       1558.75       972.25  13.30  47.700000   
2       1140.00    114.0       1554.50      1074.00  11.90  53.975000   
3       1092.00    122.0       1583.75      1203.25  11.00  60.000000   
4       1205.00    116.0       1490.00      1110.00  11.15  59.575001   

         AH            Datetime  
0  0.757754 2004-03-10 18:00:00  
1  0.725487 2004-03-10 19:00:00  
2  0.750239 2004-03-10 20:00:00 

In [33]:
# Data Integration
# Simulate integration

df_part1 = df[['Datetime', 'CO(GT)', 'NOx(GT)']]
df_part2 = df[['Datetime', 'NO2(GT)', 'T', 'RH']]

# Merge
merged_df = pd.merge(df_part1, df_part2, on='Datetime')

print(merged_df.head())

             Datetime  CO(GT)  NOx(GT)  NO2(GT)      T         RH
0 2004-03-10 18:00:00     2.6    166.0    113.0  13.60  48.875001
1 2004-03-10 19:00:00     2.0    103.0     92.0  13.30  47.700000
2 2004-03-10 20:00:00     2.2    131.0    114.0  11.90  53.975000
3 2004-03-10 21:00:00     2.2    172.0    122.0  11.00  60.000000
4 2004-03-10 22:00:00     1.6    131.0    116.0  11.15  59.575001


In [40]:
# Error Correcting
# Ensure no negative pollutant values
pollutants = ['CO(GT)', 'NOx(GT)', 'NO2(GT)']

for col in pollutants:
    df[col] = df[col].apply(lambda x: x if x >= 0 else np.nan)

# Fill again after correction
df.fillna(df.mean(numeric_only=True), inplace=True)


/tmp/ipykernel_4951/993177152.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].apply(lambda x: x if x >= 0 else np.nan)
/tmp/ipykernel_4951/993177152.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.fillna(df.mean(numeric_only=True), inplace=True)


In [35]:
# Data Transformation
from sklearn.preprocessing import StandardScaler

# Extract time features
df['Hour'] = df['Datetime'].dt.hour
df['Month'] = df['Datetime'].dt.month

# Scaling
scaler = StandardScaler()
num_cols = df.select_dtypes(include=np.number).columns

df[num_cols] = scaler.fit_transform(df[num_cols])

print(df.head())

       CO(GT)  PT08.S1(CO)      NMHC(GT)  C6H6(GT)  PT08.S2(NMHC)   NOx(GT)  \
39   0.358515     3.013363 -2.842171e-14  2.507322       2.168178  0.354115   
184  3.063565     3.455236 -2.842171e-14  2.365002       2.070922  1.242642   
185  1.104428     2.573025 -2.842171e-14  1.055411       1.093618  0.155027   
186  0.412968     1.994602 -2.842171e-14  0.721854       0.813711 -0.306122   
187  0.412968     2.036027 -2.842171e-14  0.660412       0.760339 -0.497542   

     PT08.S3(NOx)   NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)         T        RH  \
39      -0.361039  0.187059      2.035819     2.093670 -1.126309  0.462382   
184     -0.792678  1.249654      1.883108     2.852524 -0.165363 -0.447845   
185     -0.026056  1.307209      1.062286     2.027897  0.163308 -0.800613   
186      0.335408  0.933101      0.777544     1.389078  0.503121 -1.164994   
187      0.294363  0.760436      0.731412     1.040482  0.536545 -1.135960   

           AH            Datetime      Hour     Month  


/tmp/ipykernel_4951/4123566483.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Hour'] = df['Datetime'].dt.hour
/tmp/ipykernel_4951/4123566483.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Month'] = df['Datetime'].dt.month
/tmp/ipykernel_4951/4123566483.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/us

In [41]:
# Data Model Building
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Features & target
X = df.drop(columns=['CO(GT)', 'Datetime'])
y = df['CO(GT)']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = LinearRegression()
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)

MSE: 0.1324542393441942
